In [1]:
from pathlib import Path
from metasmith.python_api import Agent, Source, Std, DataInstanceLibrary, WorkflowTask
from local.constants import WORKSPACE_ROOT

dtypes, containers, transforms = Std()

path_to_agent_home = Path("./cache/local_home").resolve()
smith = Agent(
    home = Source.FromLocal(path_to_agent_home),
)
# smith.Deploy()

In [2]:
# dtypes.types

In [3]:
inputs = DataInstanceLibrary("./cache/read_mapping.xgdb")
inputs.AddTypeLibrary("std", dtypes)
inputs.Add(
    [
        (WORKSPACE_ROOT/"data/raw/read_mapping/Southern_Ocean_St365_0-1um.fna", "reads.fna", "std::short_reads"),
        (WORKSPACE_ROOT/"data/raw/read_mapping/SRR8426932_metawrap_1_1000bp_rm.fa", "asm.fna", "std::assembly"),
    ]
)
inputs.Save()

In [4]:
inputs = DataInstanceLibrary.Load("./cache/read_mapping.xgdb")
for p, n, e in inputs.Iterate():
    print(n, p, e, e.parents)

std::assembly asm.fna <{data:Sequence assembly}:BD4mNSAh> set()
std::short_reads reads.fna <{data:Short sequence,format:Sequence file}:M37WupEI> set()


In [5]:
task = smith.GenerateWorkflow(
    given      = [containers, inputs],
    transforms = [transforms],
    targets    = [dtypes["per_contig_coverage"]]
)

for step in [s for p in task.plans for s in p.steps]:
    print(step.order, step.transform.name)
task.plans[0].RenderDAG("./cache/dag")

1 minimap_short
2 bedtools_genomcov
